# Module 6 / Class 6 LAB -- Overfitting Diagnosis and Regularization

**Objectives:**
- Observe overfitting on a small CIFAR-10 subset
- Apply three regularization techniques: Dropout, Early Stopping, Data Augmentation
- Compare training curves side by side
- Determine which technique helps most

**Runtime:** Use GPU (Runtime > Change runtime type > GPU).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

np.random.seed(42)
tf.random.set_seed(42)

## 1. Load CIFAR-10 (Small Subset)

We deliberately use only 5000 training samples to force overfitting.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Use only 5000 training samples
N_TRAIN = 5000
indices = np.random.choice(len(X_train_full), N_TRAIN, replace=False)
X_train = X_train_full[indices]
y_train = y_train_full[indices]

# Normalize to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training set:  {X_train.shape}")
print(f"Test set:      {X_test.shape}")

In [ ]:
# Show sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(class_names[y_train[i][0]])
    ax.axis('off')
plt.suptitle('Sample CIFAR-10 Images')
plt.tight_layout()
plt.show()

## 2. Helper Functions

In [ ]:
EPOCHS = 30
BATCH_SIZE = 64

# Store all histories for final comparison
all_histories = {}
all_test_accs = {}


def plot_curves(history, title):
    """Plot training and validation accuracy/loss."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0].set_title(f'{title} -- Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], label='Train', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[1].set_title(f'{title} -- Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 3. Variant A -- No Regularization (Baseline)

This model will overfit: training accuracy climbs high while validation accuracy stagnates or drops.

In [ ]:
def build_cnn_baseline():
    return keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])


model_a = build_cnn_baseline()
model_a.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Variant A -- No Regularization")
history_a = model_a.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    verbose=1,
)

test_loss_a, test_acc_a = model_a.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc_a:.4f}")

all_histories['A: No Regularization'] = history_a
all_test_accs['A: No Regularization'] = test_acc_a

plot_curves(history_a, 'A: No Regularization')

Notice the gap between training and validation accuracy. That gap is overfitting.

## 4. Variant B -- With Dropout

In [ ]:
def build_cnn_dropout():
    return keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ])


model_b = build_cnn_dropout()
model_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Variant B -- With Dropout")
history_b = model_b.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    verbose=1,
)

test_loss_b, test_acc_b = model_b.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc_b:.4f}")

all_histories['B: Dropout'] = history_b
all_test_accs['B: Dropout'] = test_acc_b

plot_curves(history_b, 'B: Dropout')

## 5. Variant C -- With Early Stopping

In [ ]:
model_c = build_cnn_baseline()  # Same architecture as A, no dropout
model_c.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print("Variant C -- With Early Stopping")
history_c = model_c.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1,
)

test_loss_c, test_acc_c = model_c.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc_c:.4f}")
print(f"Stopped at epoch: {len(history_c.history['loss'])}")

all_histories['C: Early Stopping'] = history_c
all_test_accs['C: Early Stopping'] = test_acc_c

plot_curves(history_c, 'C: Early Stopping')

## 6. Variant D -- With Data Augmentation

In [ ]:
# Data augmentation layers
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Show augmented examples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
sample_image = X_train[0:1]  # shape (1, 32, 32, 3)
for i, ax in enumerate(axes.flat):
    augmented = data_augmentation(sample_image, training=True)
    ax.imshow(augmented[0].numpy())
    ax.axis('off')
plt.suptitle('Data Augmentation Examples (same source image)')
plt.tight_layout()
plt.show()

In [ ]:
def build_cnn_augmented():
    return keras.Sequential([
        # Augmentation layers (only active during training)
        layers.RandomFlip('horizontal', input_shape=(32, 32, 3)),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        # Same CNN architecture
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])


model_d = build_cnn_augmented()
model_d.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Variant D -- With Data Augmentation")
history_d = model_d.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    verbose=1,
)

test_loss_d, test_acc_d = model_d.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc_d:.4f}")

all_histories['D: Data Augmentation'] = history_d
all_test_accs['D: Data Augmentation'] = test_acc_d

plot_curves(history_d, 'D: Data Augmentation')

## 7. Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for (name, hist), color in zip(all_histories.items(), colors):
    axes[0].plot(hist.history['val_accuracy'], label=name, linewidth=2, color=color)
    axes[1].plot(hist.history['val_loss'], label=name, linewidth=2, color=color)

axes[0].set_title('Validation Accuracy -- All Variants')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Loss -- All Variants')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Overfitting gap: train acc - val acc at final epoch
print("=" * 60)
print(f"{'Variant':<25} {'Test Acc':>10} {'Overfit Gap':>12}")
print("=" * 60)
for (name, hist), (_, acc) in zip(all_histories.items(), all_test_accs.items()):
    train_final = hist.history['accuracy'][-1]
    val_final = hist.history['val_accuracy'][-1]
    gap = train_final - val_final
    print(f"{name:<25} {acc:>10.4f} {gap:>12.4f}")
print("=" * 60)

---

## TODO: Student Work

### Document which technique helped most and why

Answer the following:

1. Which variant achieved the best test accuracy?
2. Which variant had the smallest overfitting gap?
3. Explain how each technique fights overfitting (in your own words):
   - Dropout
   - Early Stopping
   - Data Augmentation
4. If you could combine techniques, which combination would you try? Why?
5. Would these results change if we used all 50,000 training samples instead of 5,000?

# TODO: Write your analysis

**1. Best test accuracy:**
- ...

**2. Smallest overfitting gap:**
- ...

**3. How each technique works:**
- Dropout: ...
- Early Stopping: ...
- Data Augmentation: ...

**4. Best combination:**
- ...

**5. Effect of more data:**
- ...